In [8]:
# Install required packages for reading BXSF files and 3D plotting
import subprocess
import sys

packages = ['plotly', 'pymatgen', 'scikit-image']
for package in packages:
    try:
        __import__(package)
        print(f"{package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

plotly is already installed
pymatgen is already installed
Installing scikit-image...


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import re
import os

In [10]:
import numpy as np
import plotly.graph_objects as go
from skimage import measure

def read_bxsf_fermi_surface(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        if "Fermi Energy" in line:
            fermi_energy = float(line.strip().split()[-1])
            break

    band_block_start = lines.index('  BEGIN_BANDGRID_3D_fermi\n')
    num_bands = int(lines[band_block_start + 1].strip())
    nx, ny, nz = map(int, lines[band_block_start + 2].strip().split())

    band_data = []
    index = band_block_start + 7

    for band in range(num_bands):
        while index < len(lines) and not lines[index].strip().startswith('BAND:'):
            index += 1
        index += 1

        values = []
        while index < len(lines):
            line = lines[index].strip()
            if line.startswith('BAND:') or line.startswith('END'):
                break
            values.extend([float(x) for x in line.split()])
            index += 1

        band_data.append(np.array(values).reshape((nx, ny, nz)))

    return np.array(band_data), fermi_energy, (nx, ny, nz)

# === Usage ===
band_data_up, fermi_energy_up, grid_shape_up = read_bxsf_fermi_surface("RuO2.up.bxsf")
band_data_down, fermi_energy_down, grid_shape_down = read_bxsf_fermi_surface("RuO2.down.bxsf")

In [11]:
from plotly.subplots import make_subplots

# Load both spin channels
band_data_up, fermi_energy_up, grid_shape_up = read_bxsf_fermi_surface("RuO2.up.bxsf")
band_data_down, fermi_energy_down, grid_shape_down = read_bxsf_fermi_surface("RuO2.down.bxsf")

# Create subplots
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],
                    subplot_titles=("Spin Up", "Spin Down"))

for spin, (band_data, fermi_energy, grid_shape, col, title) in enumerate([
    (band_data_up, fermi_energy_up, grid_shape_up, 1, "Spin Up"),
    (band_data_down, fermi_energy_down, grid_shape_down, 2, "Spin Down")
]):
    nx, ny, nz = grid_shape
    repeat = 2
    for idx in [15, 16]:  # or your desired band indices
        surface = band_data[idx] - fermi_energy
        surface_tiled = np.tile(surface, (repeat, repeat, repeat))
        nx_t, ny_t, nz_t = surface_tiled.shape

        # Grid from 0 to 2 (2 BZs)
        x = np.linspace(-1.0, repeat * 1.0/2. - 1.0, nx_t)
        y = np.linspace(-1.0, repeat * 1.0/2. - 1.0, ny_t)
        z = np.linspace(-1.0, repeat * 1.0/2. - 1.0, nz_t)

        verts, faces, _, _ = measure.marching_cubes(surface_tiled, level=0.0)
        verts[:, 0] = verts[:, 0] / (nx_t - 1) * (repeat * 1.0)
        verts[:, 1] = verts[:, 1] / (ny_t - 1) * (repeat * 1.0)
        verts[:, 2] = verts[:, 2] / (nz_t - 1) * (repeat * 1.0)
        xv, yv, zv = verts.T
        # Adjust coordinates to be in the range [-1, 1]
        xv = xv / (repeat * 1.0) * 2.0 - 1.0
        yv = yv / (repeat * 1.0) * 2.0 - 1.0
        zv = zv / (repeat * 1.0) * 2.0 - 1.0
        i_faces, j_faces, k_faces = faces.T

        mesh = go.Mesh3d(
            x=xv, y=yv, z=zv,
            i=i_faces, j=j_faces, k=k_faces,
            opacity=0.6,
            color='teal',
            name=f'Band {idx + 1}',
            showscale=False
        )
        fig.add_trace(mesh, row=1, col=col)

    fig.update_scenes(
        xaxis=dict(title='kx (fractional)', range=[-0.5, 0.5]),
        yaxis=dict(title='ky (fractional)', range=[-0.5, 0.5]),
        zaxis=dict(title='kz (fractional)', range=[-0.5, 0.5]),
        aspectmode='cube',
        row=1, col=col
    )

fig.update_layout(
    width=1600,
    height=700
)
fig.show()